In [ ]:

# RQ2: SYNTHESIS NEUTRALITY - FAVOR VS OPPOSE

!pip install groq

import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import pandas as pd
from collections import defaultdict, Counter
import re
import time
import json
import os
from tqdm import tqdm


PERSIST_DIR = "/content/drive/MyDrive/arxiv-rag-project/data2/chroma_db"
COLLECTION_NAME = "cs_papers_v4"
RESULTS_DIR = "/content/drive/MyDrive/arxiv-rag-project/results"

groq_client = Groq(api_key="")

client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = client.get_collection(COLLECTION_NAME)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")



TEST_QUERIES_15 = [
    "Do transformers consistently outperform convolutional neural networks in computer vision tasks?",
    "Is reinforcement learning suitable for real-world robotic control systems?",
    "Does transfer learning risk propagating biases across different domains?",
    "Are generative adversarial networks reliable for producing realistic images?",
    "Should large language models be fine-tuned for domain-specific applications?",
    "Do pre-trained language models exhibit systemic biases against certain demographic groups?",
    "Are graph neural networks effective for all types of structured data?",
    "Is meta-learning a practical approach for few-shot learning in production systems?",
    "Should event-driven architectures replace synchronous pipelines in enterprise applications?",
    "Are Bloom filters a reliable optimization technique for high-throughput distributed systems?",
    "Do self-supervised learning methods consistently improve model generalization?",
    "Are convolutional neural networks sufficient for advanced image recognition tasks without transformers?",
    "Should reinforcement learning be limited in safety-critical environments due to unpredictability?",
    "Does increasing model size in large language models always lead to better performance?",
    "Are multimodal AI systems (combining vision and language) more prone to alignment issues than single-modality models?"
]


def classify_paper_stance(query, paper_text):
    """
    Classify paper stance using Groq
    Returns: 'FAVOR', 'OPPOSE', or 'NEUTRAL'
    """

    prompt = f"""You are an expert CS researcher analyzing paper perspectives.

Query: {query}

Paper abstract:
{paper_text[:600]}

Does this paper:
- FAVOR: Supports, advocates, shows positive results about the topic/method
- OPPOSE: Critiques, identifies limitations, shows problems
- NEUTRAL: Descriptive, balanced, or surveys multiple views

Answer with ONLY one word: FAVOR, OPPOSE, or NEUTRAL

Stance:"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "Answer with only one word: FAVOR, OPPOSE, or NEUTRAL"},
                {"role": "user", "content": prompt}
            ],
            max_tokens=10,
            temperature=0
        )

        stance = response.choices[0].message.content.strip().upper()

        if stance in ['FAVOR', 'OPPOSE', 'NEUTRAL']:
            return stance
        else:
            return 'NEUTRAL'

    except Exception as e:
        return 'NEUTRAL'



def synthesize_answer(query, papers_with_stances):
    """Synthesize answer with Groq"""

    papers_text = ""
    for i, paper in enumerate(papers_with_stances, 1):
        papers_text += f"\n[{i}] Institution: {paper['institution']}\n"
        papers_text += f"    Stance: {paper['stance']}\n"
        papers_text += f"    Text: {paper['text'][:350]}\n"

    prompt = f"""Based on these research papers, answer the question using citations [1], [2], etc.

Question: {query}

Papers:
{papers_text}

Provide a 2-3 paragraph answer with citations. Include different perspectives if present.

Answer:"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"  Error: {e}")
        return ""

# FUNCTION 3: EXTRACT CITATIONS


def extract_citations(answer_text):
    """Extract [N] from answer"""
    nums = re.findall(r'\[(\d+)\]', answer_text)
    return list(set(int(n) for n in nums if 1 <= int(n) <= 10))



print("RQ2: SYNTHESIS NEUTRALITY (15 queries)")


rq2_results = []
stance_stats = defaultdict(lambda: {'retrieved': 0, 'cited': 0})

for query_idx, query in enumerate(tqdm(TEST_QUERIES_15, desc="RQ2"), 1):
    print(f"\n[{query_idx}/15] {query[:50]}...")

    # Retrieve
    query_embedding = embed_model.encode([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10
    )

    # Classify stances
    papers_with_stances = []

    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        stance = classify_paper_stance(query, doc)

        papers_with_stances.append({
            'rank': i + 1,
            'text': doc,
            'institution': meta.get('primary_institution'),
            'privilege': meta.get('privilege'),
            'stance': stance
        })

        stance_stats[stance]['retrieved'] += 1

        time.sleep(1)  # Groq rate limit

    stance_dist = Counter(p['stance'] for p in papers_with_stances)
    print(f"  Stances: F={stance_dist['FAVOR']}, O={stance_dist['OPPOSE']}, N={stance_dist['NEUTRAL']}")

    # Synthesize
    llm_answer = synthesize_answer(query, papers_with_stances)

    time.sleep(2)

    # Extract citations
    cited_nums = extract_citations(llm_answer)
    print(f"  Cited: {cited_nums}")

    # Analyze citations
    for citation_num in cited_nums:
        idx = citation_num - 1
        if idx < len(papers_with_stances):
            cited_paper = papers_with_stances[idx]
            stance_stats[cited_paper['stance']]['cited'] += 1

            rq2_results.append({
                'query': query,
                'query_num': query_idx,
                'citation_num': citation_num,
                'institution': cited_paper['institution'],
                'privilege': cited_paper['privilege'],
                'stance': cited_paper['stance'],
                'rank': cited_paper['rank']
            })


rq2_df = pd.DataFrame(rq2_results)

print("\n" + "="*60)
print("RQ2 FINAL RESULTS")
print("="*60)

print(f"\n STANCE ANALYSIS:")
print(f"{'Stance':<12} {'Retrieved':>10} {'Cited':>8} {'Rate':>10}")
print("-"*42)

for stance in ['FAVOR', 'OPPOSE', 'NEUTRAL']:
    retr = stance_stats[stance]['retrieved']
    cite = stance_stats[stance]['cited']
    rate = (cite / retr * 100) if retr > 0 else 0
    print(f"{stance:<12} {retr:>10} {cite:>8} {rate:>9.0f}%")

# Disparity
favor_retrieved = stance_stats['FAVOR']['retrieved']
favor_cited = stance_stats['FAVOR']['cited']
oppose_retrieved = stance_stats['OPPOSE']['retrieved']
oppose_cited = stance_stats['OPPOSE']['cited']

favor_rate = (favor_cited / favor_retrieved) if favor_retrieved > 0 else 0
oppose_rate = (oppose_cited / oppose_retrieved) if oppose_retrieved > 0 else 0
disparity = favor_rate / oppose_rate if oppose_rate > 0 else 0

print(f"\n BIAS MEASUREMENT:")
print(f"  FAVOR citation rate:  {favor_rate:.1%}")
print(f"  OPPOSE citation rate: {oppose_rate:.1%}")
print(f"  Disparity: {disparity:.2f}× preference")

# Statistical test
from scipy.stats import chi2_contingency

if favor_cited + oppose_cited > 0:
    total_cited = favor_cited + oppose_cited
    total_retrieved = favor_retrieved + oppose_retrieved

    obs = [favor_cited, oppose_cited]
    exp_favor = total_cited * (favor_retrieved / total_retrieved)
    exp_oppose = total_cited * (oppose_retrieved / total_retrieved)
    exp = [exp_favor, exp_oppose]

    chi2, p_value, _, _ = chi2_contingency([obs, exp])

    print(f"\n Chi-square Test:")
    print(f"  χ² = {chi2:.2f}, p = {p_value:.4f}")
    print(f"  Result: {'Significant bias' if p_value < 0.05 else 'No significant bias'}")

# Save
rq2_df.to_csv(f'{RESULTS_DIR}/rq2_results_15queries2.csv', index=False)


